# BERTrend — stable themes (simple)

Minimal notebook: find **clear news themes that stay recognizable over time**, with **human-readable descriptions**.

How it works (see [BERTrend](https://github.com/rte-france/BERTrend)):

1. Split headlines into bi-weekly slices → train one BERTopic model per slice.
2. **Merge** topics across slices when centroid cosine ≥ `MIN_SIMILARITY` → stable theme IDs.
3. Rank themes by **persistence** (how many slices they survive + merge quality).
4. Label top themes with an LLM title + paragraph (falls back to keywords + exemplar headline).

Tuning for clarity: larger `min_cluster_size`, stricter merge threshold, finance stopwords, KeyBERT+MMR representations.

## 0. Setup

In [1]:
import os
from pathlib import Path

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["BERTREND_BASE_DIR"] = str(_ROOT / "notebooks" / "output" / "bertrend_base")

import lzma
import re

import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

from openai import OpenAI

from bertrend import LLM_CONFIG
from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.topic_analysis.data_structure import TopicDescription
from bertrend.topic_analysis.prompts import TOPIC_DESCRIPTION_PROMPT
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN,
    SOURCE_COLUMN,
    TEXT_COLUMN,
    TIMESTAMP_COLUMN,
    URL_COLUMN,
    group_by_days,
)
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

PROJECT_ROOT = _ROOT
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output"
MODELS_DIR = OUTPUT_DIR / "bertrend_stable_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- run config (small window for fast iteration) ---
DATE_START = "2024-01-01T00:00:00+00:00"
DATE_END = "2024-07-01T00:00:00+00:00"   # H1 2024 → ~13 bi-weekly slices
TAG = "2024H1"
SAMPLE_N = 20_000
GRANULARITY_DAYS = 14
WINDOW_SIZE = 28
MIN_TOPIC_SIZE = 20
MIN_SAMPLES = 5
MIN_SIMILARITY = 0.75                     # stricter merge → themes must truly persist
MIN_ACTIVE_SLICES = 4                     # theme must appear in ≥4 slices (~2 months)
TOP_N = 12                                # themes to describe and plot
EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
RANDOM_SEED = 42

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {DEVICE}  |  window: {DATE_START[:10]} → {DATE_END[:10]}  |  sample ≤ {SAMPLE_N:,}")

2026-06-08 16:46:16.963 | INFO     | bertrend:<module>:20 - Loaded .env file


Device: mps  |  window: 2024-01-01 → 2024-07-01  |  sample ≤ 20,000


## 1. Load & clean headlines

Light cleaning only: drop wire boilerplate, strip source prefixes, require ≥4 words.

In [2]:
BOILERPLATE_RE = re.compile(
    r"\b("
    # corporate filings / governance
    r"half[- ]year|interim (results|report)|dividend|agm|egm|"
    r"appoint(s|ed|ment)?|total voting rights|trading update|block listing|"
    r"transaction in own shares|holding\(s\) in|net asset value|notice of|"
    # earnings calendar
    r"q[1-4] (results|earnings)|fy\d{2}|quarterly (results|report)|"
    r"full[- ]year results|earnings (call|preview|release|report)|"
    # analyst rating actions
    r"price target|reiterates?|maintains rating|raises? (price )?target|"
    r"cuts? (price )?target|upgrades?|downgrades?|initiates? coverage|"
    r"outperform|underperform|overweight|underweight|"
    # market-data ticker wire
    r"adr|gdr|equity premium|premium/discount"
    r")\b",
    re.IGNORECASE,
)
FINANCE_STOP = ENGLISH_STOP_WORDS.union({
    # corporate / governance boilerplate
    "inc", "plc", "ltd", "ceo", "cfo", "coo", "results", "result", "announces",
    "announced", "dividend", "shares", "stock", "says", "said",
    "chief", "director", "executive", "secretary", "chairman", "officer", "board",
    # generic news filler
    "new", "year", "today", "week", "good", "best", "world", "global", "people",
    "city", "limited", "release", "report", "update",
    # earnings / market-data noise
    "quarter", "quarterly", "monthly", "prelim", "est", "outperform",
    "underperform", "overweight", "underweight", "raised", "raises", "nav",
    "adr", "gdr", "points",
})


def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text


with lzma.open(RAW_DIR / "raw_news_2024.csv.xz", "rb") as f:
    corpus = (
        pl.scan_csv(f, infer_schema_length=10_000)
        .select(["Headline", "CaptureTime", "WireName"])
        .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
        .filter(
            (pl.col("CaptureTime") >= pl.lit(DATE_START).str.to_datetime(time_zone="UTC"))
            & (pl.col("CaptureTime") < pl.lit(DATE_END).str.to_datetime(time_zone="UTC"))
            & pl.col("Headline").is_not_null()
        )
        .collect()
        .unique(subset=["Headline"])
    )

df = corpus.to_pandas().rename(columns={
    "Headline": TEXT_COLUMN,
    "CaptureTime": TIMESTAMP_COLUMN,
    "WireName": SOURCE_COLUMN,
})
df = df[~df[TEXT_COLUMN].fillna("").str.contains(BOILERPLATE_RE)]
df[TEXT_COLUMN] = df[TEXT_COLUMN].map(strip_prefix)
df = df[df[TEXT_COLUMN].str.split().map(len) >= 4]
df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN]).dt.tz_localize(None)
df = df.sort_values(TIMESTAMP_COLUMN)

# Stratified cap per bi-weekly slice — random global sampling leaves thin slices.
_slice = df[TIMESTAMP_COLUMN].dt.floor(f"{GRANULARITY_DAYS}D")
_per_slice = max(300, SAMPLE_N // _slice.nunique())
df = (
    df.groupby(_slice, group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), _per_slice), random_state=RANDOM_SEED))
    .sort_values(TIMESTAMP_COLUMN)
    .reset_index(drop=True)
)
df[DOCUMENT_ID_COLUMN] = df.index
df[URL_COLUMN] = None
df[SOURCE_COLUMN] = df[SOURCE_COLUMN].fillna("unknown")

print(f"Headlines: {len(df):,}  |  span: {df[TIMESTAMP_COLUMN].min().date()} → {df[TIMESTAMP_COLUMN].max().date()}")
df.head(3)

Headlines: 19,992  |  span: 2024-01-01 → 2024-06-30


,text,timestamp,source,document_id,url
0,Publication Annual Report 2023,2024-01-01 00:05:51.168,CO5,0,None
1,LensToLens | How a Chinese NEV contributes to ...,2024-01-01 00:11:42.379,NS6,1,None
2,PGA Tour seeks extension on commercial deal wi...,2024-01-01 00:12:07.861,NS1,2,None


## 2. Configure BERTrend & train

One embedding pass, then BERTrend fits + merges models per slice. Larger clusters and a higher merge threshold favour **fewer, stabler** themes.

In [3]:
bertopic_config = f"""
[global]
language = "English"

[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]

[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}

[hdbscan_model]
min_cluster_size = {MIN_TOPIC_SIZE}
min_samples = {MIN_SAMPLES}
metric = "euclidean"
cluster_selection_method = "eom"
prediction_data = true

[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = 3

[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true

[mmr_model]
diversity = 0.4

[reduce_outliers]
strategy = "c-tf-idf"
"""

topic_model = BERTopicModel(bertopic_config)
topic_model.vectorizer_model = CountVectorizer(
    stop_words=list(FINANCE_STOP),
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    ngram_range=(1, 2),
    min_df=2,
)
topic_model.config["bertopic_model"]["representation_model"] = [
    KeyBERTInspired(top_n_words=20, nr_repr_docs=5, nr_candidate_words=40),
    MaximalMarginalRelevance(diversity=0.4),
]

bertrend = BERTrend(topic_model=topic_model)
bertrend.config["granularity"] = GRANULARITY_DAYS
bertrend.config["min_similarity"] = MIN_SIMILARITY

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

# BERTrend's _train_by_period() does not forward embedding_model to fit(), but
# KeyBERTInspired needs it during update_topics — patch fit() like 0.4-bertrend.
import types

def _fit_with_embed(self, docs, embeddings, embedding_model=None, **kwargs):
    return BERTopicModel.fit(self, docs, embeddings, embedding_model=embedder, **kwargs)

topic_model.fit = types.MethodType(_fit_with_embed, topic_model)

embeddings = embedder.encode(
    df[TEXT_COLUMN].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

grouped = {ts: g for ts, g in group_by_days(df, GRANULARITY_DAYS).items() if not g.empty}
print(f"Slices: {len(grouped)}  |  docs/slice median: {int(np.median([len(g) for g in grouped.values()]))}")

bertrend.train_topic_models(
    grouped_data=grouped,
    embedding_model=embedder,
    embeddings=embeddings,
    bertrend_models_path=MODELS_DIR,
    save_topic_models=True,
)
if bertrend.merged_df is None:
    raise RuntimeError(
        "No themes were merged — every slice failed. "
        "Scroll up for per-period errors (KeyBERT needs embedder in fit(); "
        "or min_cluster_size too large for a slice)."
    )

bertrend.calculate_signal_popularity()
bertrend.save_model(models_path=MODELS_DIR)
print(f"Merged themes: {bertrend.merged_df['Topic'].nunique()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Slices: 13  |  docs/slice median: 1436
2026-06-08 16:47:43.158 | INFO     | bertrend.BERTrend:train_topic_models:288 - Training topic model 1/13...


2026-06-08 16:47:43.159 | DEBUG    | bertrend.BERTrend:_train_by_period:186 - Processing period: 2024-01-01 00:00:00


2026-06-08 16:47:43.160 | DEBUG    | bertrend.BERTrend:_train_by_period:187 - Number of documents: 2479


...

2026-06-08 16:48:34.715 | INFO     | bertrend.BERTrend:train_topic_models:301 - Saving topic model for period 2024-06-17 00:00:00...


2026-06-08 16:48:34.718 | DEBUG    | bertrend.BERTrend:train_topic_models:311 - Successfully processed period: 2024-06-17 00:00:00


2026-06-08 16:48:34.718 | SUCCESS  | bertrend.BERTrend:train_topic_models:331 - Finished training all topic models


2026-06-08 16:48:35.015 | INFO     | bertrend.BERTrend:save_model:753 - BERTrend model saved to: /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_stable_models


Merged themes: 40


## 3. Pick themes that **stay in time**

Persistence = how many bi-weekly slices a theme keeps getting merged (from `all_merge_histories_df`), plus average merge cosine. We keep themes active in ≥ `MIN_ACTIVE_SLICES` slices.

In [4]:
def flatten_docs(doc_entries) -> list[str]:
    """Documents in merged_df are [(timestamp, [headline, ...]), ...]."""
    out = []
    for entry in doc_entries:
        if isinstance(entry, tuple) and len(entry) == 2:
            out.extend(d for d in entry[1] if isinstance(d, str))
        elif isinstance(entry, str):
            out.append(entry)
    return out


def keyword_list(rep) -> list[str]:
    words = rep if isinstance(rep, list) else str(rep).replace("_", " ").split()
    return [w.strip().lower() for w in words[:10] if w.strip()]


def keywords_label(rep) -> str:
    return ", ".join(keyword_list(rep)[:6])


def relevant_docs(doc_entries, rep, n: int = 5, min_words: int = 5) -> list[str]:
    """Rank a theme's headlines by overlap with its top keywords (ties → shorter).

    BERTrend's c-tf-idf outlier reduction dumps off-topic docs into the nearest
    cluster, so first/last picks are unreliable. Keyword overlap keeps the
    exemplar (and LLM context) genuinely on-theme."""
    kws = keyword_list(rep)
    docs = [d for d in flatten_docs(doc_entries) if len(d.split()) >= min_words]
    if not docs:
        docs = flatten_docs(doc_entries)
    if not docs:
        return []

    def score(h: str):
        hl = h.lower()
        return (sum(kw in hl for kw in kws), -len(h.split()))

    return sorted(docs, key=score, reverse=True)[:n]


if bertrend.merged_df is None or bertrend.all_merge_histories_df is None:
    raise RuntimeError("Run the training cell first — merged_df is empty.")

mh = bertrend.all_merge_histories_df
merged = bertrend.merged_df.set_index("Topic")
current_date = df[TIMESTAMP_COLUMN].max()
_, weak_df, strong_df = bertrend.classify_signals(WINDOW_SIZE, current_date)

rows = []
for topic_id, grp in mh.groupby("Topic1"):
    if topic_id not in merged.index:
        continue
    mrow = merged.loc[topic_id]
    active_slices = grp["Timestamp"].nunique() + 1
    if active_slices < MIN_ACTIVE_SLICES:
        continue

    rows.append({
        "theme_id": topic_id,
        "active_slices": active_slices,
        "mean_merge_sim": grp["Similarity"].mean(),
        "min_merge_sim": grp["Similarity"].min(),
        "docs_total": int(mrow["Document_Count"]),
        "keywords": keywords_label(mrow["Representation"]),
        "exemplar": next(iter(relevant_docs(mrow["Documents"], mrow["Representation"], n=1)), ""),
        "signal": "strong" if topic_id in set(strong_df["Topic"]) else (
            "weak" if topic_id in set(weak_df["Topic"]) else "noise"
        ),
    })

stable = (
    pd.DataFrame(rows)
    .sort_values(["active_slices", "docs_total"], ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)
print(f"Stable themes (≥{MIN_ACTIVE_SLICES} slices): {len(rows)} total, showing top {len(stable)}")
display(stable[["theme_id", "active_slices", "mean_merge_sim", "docs_total", "signal", "keywords", "exemplar"]])

Stable themes (≥4 slices): 24 total, showing top 12


,theme_id,active_slices,mean_merge_sim,docs_total,signal,keywords,exemplar
0,0,11,0.877948,1796,strong,"lawsuit, supreme court, judge, trial, illegal,...",Supreme Court of Canada refuses to seize Irani...
1,4,11,0.913442,1464,strong,"january, tuesday, month, announcement, confere...","Three Months Ended June 30, 2024 Materials for..."
2,7,11,0.900402,1080,strong,"benchmark, eur, usd, silver, amc, total","CRH EUR Benchmark Covered; 5Y, 10Y"
3,12,11,0.908598,984,strong,"unveils, smartphone, launches, introduces, ope...","Corero launches ""pre-emptive"" AI cybersecurity..."
4,3,11,0.866400,903,strong,"arabia, dubai, saudi arabia, korea, hong kong,...",Minister meets Cyprus’ ambassador to Saudi Arabia
5,15,11,0.915864,616,weak,"therapeutics, pharma, life sciences, fda, drug...",BridgeBio Pharma Says FDA Clears Investigation...
6,2,10,0.851439,1119,strong,"health, care, michelle, education, lgbtq, berry",‘Never forget value of values’: Co-founder and...
7,6,9,0.950722,790,weak,"accession number, notice, iii, municipal, bank...",BLACKROCK NEW YORK MUNICIPAL BOND TRUST: N-8F NTC
8,5,9,0.921580,565,noise,"xrp, options surge, bitcoin, surges, etfs, bul...",Bitcoin ETFs to Start Trading Today After Long...
9,13,8,0.866122,835,strong,"deal, acquisition, acquires, universal, subsid...",Sembcorp Industries’ Subsidiary Completes Acqu...


## 4. Realistic descriptions

Calls OpenAI directly with structured output (model + key from `code/.env`), using BERTrend's title/description prompt. We bypass BERTrend's `get_topic_description` because its `parse()` relies on `Runner.run_sync()`, which fails inside a Jupyter event loop. Each theme's 5 most keyword-relevant headlines are passed as context. Without an API key, it falls back to keywords + exemplar headline.

In [5]:
# Call OpenAI directly with structured output. We bypass BERTrend's
# get_topic_description() because its parse() uses the openai-agents Runner.run_sync(),
# which raises "event loop already running" inside a Jupyter kernel. A plain sync
# client.beta.chat.completions.parse() works fine here.
_llm = None
if LLM_CONFIG.get("api_key") and not LLM_CONFIG["api_key"].startswith("$"):
    _llm = OpenAI(api_key=LLM_CONFIG["api_key"], base_url=LLM_CONFIG.get("base_url") or None)


def describe_theme(topic_rep: str, docs_text: str) -> TopicDescription | None:
    if _llm is None:
        return None
    prompt = TOPIC_DESCRIPTION_PROMPT["en"].format(
        topic_representation=topic_rep, docs_text=docs_text
    )
    try:
        resp = _llm.beta.chat.completions.parse(
            model=LLM_CONFIG["model"],
            messages=[{"role": "user", "content": prompt}],
            response_format=TopicDescription,
            temperature=0.1,
        )
        return resp.choices[0].message.parsed
    except Exception as e:
        print(f"  LLM error for '{topic_rep[:40]}…': {e}")
        return None


catalog = []
for _, row in stable.iterrows():
    tid = int(row["theme_id"])
    mrow = merged.loc[tid]
    sample_docs = relevant_docs(mrow["Documents"], mrow["Representation"], n=5)
    topic_rep = ", ".join(keyword_list(mrow["Representation"]))
    docs_text = "\n\n".join(f"Document {i + 1}: {d}" for i, d in enumerate(sample_docs))

    desc = describe_theme(topic_rep, docs_text)
    if desc is not None:
        title, description = desc.title, desc.description
    else:
        title = row["keywords"].replace(",", " ").title()[:60]
        description = f"Recurring coverage of {row['keywords']}. Example: \"{row['exemplar']}\""

    catalog.append({
        "theme_id": tid,
        "title": title,
        "description": description,
        "active_slices": row["active_slices"],
        "mean_merge_sim": round(row["mean_merge_sim"], 3),
        "docs_total": row["docs_total"],
        "signal": row["signal"],
        "keywords": row["keywords"],
        "exemplar": row["exemplar"],
    })
    print(f"[{tid}] {title}")
    print(f"     {description[:200]}{'…' if len(description) > 200 else ''}\n")

catalog_df = pd.DataFrame(catalog)
out_parquet = OUTPUT_DIR / f"bertrend_stable_themes_{TAG}.parquet"
catalog_df.to_parquet(out_parquet, index=False)
print(f"Saved → {out_parquet}")

[0] Supreme Court Cases Involving State Lawsuits
     Recent developments highlight the Supreme Court's involvement in significant lawsuits concerning state regulations and political actions. Notably, 27 Republican states are urging the Court to review a…



[4] January Conference Announcements and Meetings
     January 2024 features significant events including earnings conference calls and financial announcements. Notably, a Q4 2023 earnings call is scheduled for January 23, highlighting corporate financial…



[7] EUR Benchmark Covered Bonds Overview
     The topic focuses on EUR benchmark covered bonds, highlighting various issuances and their characteristics. Key examples include CRH's 5Y and 10Y covered bonds, Aareal Bank's 4Y long covered bond, and…



[12] Recent Innovations in AI and Digital Platforms
     Recent developments in artificial intelligence and digital platforms have showcased significant advancements. OpenAI has introduced GPT-4o, an AI model that promises enhanced performance and efficienc…



[3] International Diplomatic Visits and Relations
     Recent diplomatic activities highlight the interactions between Saudi Arabia and various nations, including Cyprus and Australia, as ministers engage in discussions to strengthen bilateral relations. …



...

[6] Municipal Finance and Investment Insights
     The topic encompasses various aspects of municipal finance, particularly focusing on investment vehicles such as municipal bond trusts and income funds. Key entities include BlackRock, which is promin…



[5] Bitcoin and XRP Surge Amid ETF Developments
     Recent developments in the cryptocurrency market have led to significant surges in Bitcoin and XRP prices. Following the SEC's approval of Bitcoin ETFs, trading commenced, resulting in a notable influ…



[13] Recent Acquisitions in Realty and Renewables
     Recent developments in the business landscape highlight a series of strategic acquisitions and partnerships, particularly in the real estate and renewable energy sectors. Sembcorp Industries' subsidia…



[8] Recent Insider Stock Sales Overview
     Recent filings reveal significant insider stock sales across various companies, highlighting trends in executive trading behavior. Notable transactions include Officer Golan's sale of $1 million in Ta…



[10] Recent Airline Incidents and Flight Delays
     Recent events have highlighted significant safety concerns and operational disruptions within the airline industry. Delta Airlines faced multiple flight incidents over a short period, raising question…

Saved → /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_stable_themes_2024H1.parquet


## 5. Timeline — do they really stay?

Popularity per slice for the stable themes. Flat or rising lines = persistent attention; sharp single spikes = probably noise.

In [6]:
bertrend.calculate_signal_popularity()
timeline_rows = []
for tid in stable["theme_id"]:
    ts_data = bertrend.topic_sizes[tid]
    label = catalog_df.loc[catalog_df["theme_id"] == tid, "title"].iloc[0]
    for ts, pop in zip(ts_data["Timestamps"], ts_data["Popularity"]):
        timeline_rows.append({"theme_id": tid, "title": label, "timestamp": ts, "popularity": pop})

timeline = pd.DataFrame(timeline_rows)
fig = px.line(
    timeline,
    x="timestamp", y="popularity", color="title",
    title=f"Stable theme popularity over time ({TAG})",
    labels={"popularity": "attention (docs + decay)", "title": "theme"},
)
fig.update_layout(legend_title_text="theme", height=520)
out_html = OUTPUT_DIR / f"bertrend_stable_themes_{TAG}.html"
fig.write_html(out_html)
fig.show()
print(f"Saved → {out_html}")

Saved → /Users/federicocinus/Progetti - Local/ThematicTrading/code/notebooks/output/bertrend_stable_themes_2024H1.html


In [ ]:
# TODO Experiment

Clean energy: 
- peak [2020-2021]
- detection interval: [19-21] 
- filter bloomberg